# Sprint 4 — Contextual Relevancy evaluation (DeepEval)

Retrieves chunks for the golden questions and scores how relevant the retrieved context is to each question with DeepEval's `ContextualRelevancyMetric`. This evaluates retrieval directly; it does not generate chatbot answers.

The notebook reports a score from 0 to 1, the judge's reason, and whether each question meets the project's 0.9 target. It also saves a Markdown summary to `tests/eval/contextual_relevancy_results.md`.

**Before running:** Set `OPEN_AI_API_KEY` (or `OPENAI_API_KEY`) in `.env` and have the persisted ChromaDB in `vectorstore/`.

## 1. Setup

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env", override=True)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPEN_AI_API_KEY", "")
if not os.environ["OPENAI_API_KEY"]:
    raise RuntimeError("No OpenAI API key found. Set OPEN_AI_API_KEY in .env.")

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

from src.evaluation.contextual_relevancy import score_contextual_relevancy
from src.retrieval.retriever import retrieve

## 2. Evaluation config

In [ ]:
THRESHOLD = 0.9
RESULTS_PATH = ROOT / "tests" / "eval" / "contextual_relevancy_results.md"

## 3. Golden questions

These are the questions used by the existing Sprint 4 Faithfulness evaluation.

In [ ]:
GOLDEN_QUESTIONS = [
    ("Q1", "What are the key safety requirements when working at height?"),
    ("Q2", "What safety precautions should be followed when working on roofs?"),
    ("Q3", "What controls should be used to prevent falls when working at height?"),
    ("Q4", "What are the main safety requirements for scaffolding in New Zealand?"),
    ("Q5", "What safety considerations apply when working with or near asbestos?"),
    ("Q6", "What should be considered when managing asbestos in a building?"),
    ("Q7", "What is required when conducting an asbestos survey?"),
    ("Q8", "What safety controls should be used when carrying out excavation work?"),
    ("Q9", "What are the main safety risks associated with workplace machinery?"),
    ("Q10", "What safety precautions should be followed when using mobile elevating work platforms?"),
    ("Q11", "What are the safety requirements for using safety nets?"),
    ("Q12", "What should workers consider when managing workplace traffic?"),
    ("Q13", "What health and safety considerations apply to industrial rope access?"),
    ("Q14", "How should risks from workplace exposure be monitored and managed?"),
    ("Q15", "A construction site has workers operating machinery near an excavation. What controls should be considered to keep workers safe?"),
]

## 4. Retrieve and score

Only the question and retrieved chunk text are sent to the Contextual Relevancy judge. No answer generation is run.

In [ ]:
runs = []
for question_id, question in GOLDEN_QUESTIONS:
    chunks = retrieve(question)
    result = score_contextual_relevancy(
        question,
        [chunk["text"] for chunk in chunks],
        threshold=THRESHOLD,
    )
    runs.append({
        "id": question_id,
        "question": question,
        "chunk_count": len(chunks),
        **result,
    })
    print(
        f"{question_id}: {len(chunks)} chunks, "
        f"score={result['score']:.3f}, "
        f"{'Pass' if result['passed'] else 'Fail'}"
    )

## 5. Summary and saved report

In [ ]:
from statistics import mean

mean_score = mean(run["score"] for run in runs)
passed_count = sum(run["passed"] for run in runs)
print(f"Mean Contextual Relevancy: {mean_score:.3f}")
print(f"Pass rate (>= {THRESHOLD}): {passed_count}/{len(runs)}")

def _cell(value):
    return str(value).replace("|", "\\|").replace("\n", " ")

lines = [
    "# Contextual Relevancy results",
    "",
    f"- Run: {datetime.now().astimezone().strftime('%Y-%m-%d %H:%M')}",
    "- Judge model: `gpt-4.1`",
    f"- Threshold: {THRESHOLD}",
    f"- Mean Contextual Relevancy: {mean_score:.3f} over {len(runs)} question(s)",
    f"- Pass rate: {passed_count}/{len(runs)}",
    "",
    "| ID | Question | Chunks | Status | Score | Reason |",
    "|---|---|---:|---|---:|---|",
]
for run in runs:
    status = "Pass" if run["passed"] else "Fail"
    lines.append(
        f"| {run['id']} | {_cell(run['question'])} | {run['chunk_count']} "
        f"| {status} | {run['score']:.3f} | {_cell(run['reason'])} |"
    )

RESULTS_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Wrote {RESULTS_PATH.relative_to(ROOT)}")